# KNN

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def train_knn(X, y): 

    neigborhoods = [3, 5, 7, 11, 13]
    weights = ['uniform', 'distance']

    rows = []
    for n in neigborhoods: 
        for w in weights:
            
            accs_train = []
            accs_val = []
            f1_scores = []
            recall_scores = []
            
            kf = KFold(n_splits=5)
            
            for train_index, test_index in kf.split(X): 
                xtrain, xtest = X[train_index], X[test_index]
                ytrain, ytest = y[train_index], y[test_index]
                
                scaler = StandardScaler() 
                xtrain, xtest = scaler.fit_transform(xtrain), scaler.transform(xtest)

                model = KNeighborsClassifier(
                    p=2, 
                    weights=w, 
                    n_neighbors=n
                )

                model.fit(xtrain, ytrain)

                predictions_train = model.predict(xtrain)
                predictions_val = model.predict(xtest)
                
                accs_train.append(accuracy_score(ytrain, predictions_train))
                accs_val.append(accuracy_score(ytest, predictions_val))
                f1_scores.append(f1_score(ytest, predictions_val))
                recall_scores.append(recall_score(ytest, predictions_val))
                
            rows.append({
                'numero de vecinos': n,
                'pesos': w,
                'media acc train': np.mean(accs_train), 
                'media acc validacion': np.mean(accs_val),
                'IC acc validacion': np.std(accs_val),
                'media f1': np.mean(f1_scores),
                "IC f1": np.std(f1_scores),
                'media recall': np.mean(recall_scores),
                'IC recall': np.std(recall_scores)
            })
    
    results = pd.DataFrame(rows)
    return results.sort_values(by='media f1', ascending=False)

In [3]:
# Con los datos sin filtrado 

from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() # 20% datos para validacion por defecto 

train_knn(X_train, y_train)

,numero de vecinos,pesos,media acc train,media acc validacion,IC acc validacion,media f1,IC f1,media recall,IC recall
1,3,distance,1.000000,0.954375,0.005604,0.954930,0.005568,0.960834,0.008161
0,3,uniform,0.973406,0.952375,0.006116,0.952892,0.006111,0.957590,0.007543
3,5,distance,1.000000,0.951625,0.004268,0.952169,0.004343,0.957366,0.008916
5,7,distance,1.000000,0.949000,0.004287,0.949689,0.004387,0.957113,0.008403
2,5,uniform,0.962656,0.947625,0.005084,0.948064,0.005239,0.950638,0.009036
7,11,distance,1.000000,0.945750,0.003921,0.946513,0.003843,0.954145,0.008556
9,13,distance,1.000000,0.943750,0.005576,0.944480,0.005686,0.951381,0.009272
4,7,uniform,0.957156,0.943500,0.004194,0.944052,0.004243,0.947684,0.008986
6,11,uniform,0.948094,0.937375,0.004209,0.937872,0.004027,0.939498,0.008834
8,13,uniform,0.944094,0.935000,0.006162,0.935365,0.006192,0.934987,0.008568


In [ ]:

# Con los datos filtrados 

from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned() # 20% datos para validacion por defecto 


train_knn(X_train, y_train)

,numero de vecinos,pesos,media acc train,media acc validacion,IC acc validacion,media f1,IC f1,media recall,IC recall
1,3,distance,1.000000,0.951625,0.005009,0.951987,0.004546,0.963586,0.002791
3,5,distance,1.000000,0.950875,0.005557,0.951099,0.005227,0.960049,0.002342
0,3,uniform,0.971000,0.948250,0.005469,0.948463,0.005033,0.956805,0.003134
5,7,distance,1.000000,0.948125,0.006435,0.948379,0.006039,0.957518,0.002608
2,5,uniform,0.960031,0.946375,0.005828,0.946471,0.005614,0.953004,0.004000
7,11,distance,1.000000,0.945750,0.009349,0.946028,0.008911,0.955024,0.006669
9,13,distance,1.000000,0.943500,0.009887,0.943737,0.009420,0.951727,0.006703
4,7,uniform,0.954406,0.943375,0.006785,0.943486,0.006353,0.949730,0.003706
6,11,uniform,0.946500,0.938000,0.009710,0.937905,0.009499,0.941173,0.009153
8,13,uniform,0.943469,0.934625,0.010305,0.934526,0.010084,0.937876,0.008817


Finalmente se escoge el modelo que mejor puntuacion saco, KNN con pesos basados en la distanca y 11 vecinos

In [3]:
from sklearn.decomposition import PCA
from libs.linear_reduction import train_models 
from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned()

models = [
    KNeighborsClassifier(
        weights='distance', 
        n_neighbors=3
    ), 
    KNeighborsClassifier(
            weights='distance', 
            n_neighbors=5
    )
]

scaler = StandardScaler()
xtrain, xtest = scaler.fit_transform(X_train), scaler.transform(X_test)

pca = PCA(n_components=0.95)
xtrain, xtest = pca.fit_transform(xtrain), pca.transform(xtest)

train_models(xtrain, xtest, y_train, y_test, models, ['weights', 'n_neighbors'])



,weights,n_neighbors,acc train,acc test,f1_score,recall
0,distance,3,1.0,0.9495,0.950123,0.964895
1,distance,5,1.0,0.9525,0.952994,0.965898


In [ ]:
from libs.linear_reduction import validate_umap_reduction

xtrain, xtest, ytrain, ytest = get_filtered_data_partitioned()

validate_umap_reduction(xtrain, ytrain, models[0])

/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces di

,n_components,n_neighbors,min_dist,mean_train_acc,mean_test_acc,ic_acc,mean_f1,ic_f1,mean_recall,ic_recall
21,20,15,0.1,1.0,0.921773,0.003594,0.918762,0.003811,0.926877,0.005154
19,20,10,0.1,1.0,0.918644,0.005398,0.915933,0.005688,0.928588,0.004276
7,5,10,0.1,1.0,0.918123,0.005525,0.914849,0.006997,0.922642,0.006305
13,10,10,0.1,1.0,0.917992,0.005382,0.914848,0.007459,0.924556,0.003388
9,5,15,0.1,1.0,0.917992,0.007062,0.914739,0.008367,0.922864,0.008769
15,10,15,0.1,1.0,0.917471,0.003989,0.914282,0.005485,0.922964,0.003394
6,5,10,0.0,1.0,0.916819,0.002166,0.913718,0.003742,0.923344,0.003920
12,10,10,0.0,1.0,0.916428,0.009305,0.913460,0.010665,0.925628,0.008043
14,10,15,0.0,1.0,0.916428,0.006285,0.913429,0.007432,0.924666,0.004936
8,5,15,0.0,1.0,0.914993,0.011318,0.911708,0.013735,0.922740,0.010760


In [5]:
validate_umap_reduction(xtrain, ytrain, models[1])

/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces di

,n_components,n_neighbors,min_dist,mean_train_acc,mean_test_acc,ic_acc,mean_f1,ic_f1,mean_recall,ic_recall
15,10,15,0.1,1.0,0.921382,0.010003,0.918435,0.012445,0.930910,0.011328
9,5,15,0.1,1.0,0.920600,0.006162,0.917471,0.008266,0.926710,0.008524
19,20,10,0.1,1.0,0.919426,0.007620,0.916824,0.009186,0.931989,0.002181
21,20,15,0.1,1.0,0.919166,0.010006,0.916053,0.011437,0.925883,0.011386
7,5,10,0.1,1.0,0.918644,0.007913,0.915456,0.009863,0.924769,0.005842
13,10,10,0.1,1.0,0.918123,0.008096,0.915209,0.009233,0.926870,0.009319
14,10,15,0.0,1.0,0.917340,0.008342,0.914107,0.010234,0.923701,0.010952
23,20,30,0.1,1.0,0.916297,0.008138,0.913337,0.009987,0.926106,0.012914
6,5,10,0.0,1.0,0.916037,0.007110,0.913229,0.008201,0.926564,0.005727
20,20,15,0.0,1.0,0.916297,0.013351,0.913039,0.015201,0.923629,0.016682
